# 🎯 [QLT04_M04] 회귀 분석 및 원인 규명 모델 기반 가동률 저하 핵심 병목 요인 정밀 분석 엔진 고도화 (훈련생 실습용)
### 훈련생 실습 워크북 (Google Colab / Jupyter Student Edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/QLT04/qlt04_m04_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 품질·규격·검사 (Level 2) |
| **미션 ID** | QLT04_M04 |
| **미션 주제** | 회귀 분석 및 원인 규명 모델 기반 가동률 저하 핵심 병목 요인 정밀 분석 엔진 고도화 |
| **소요 시간** | 약 3시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
외주정비는 정비 대기, 외주 검사 대기, 재작업, 규격·제원 불일치가 겹치면서 장비 가동률을 떨어뜨리지만 요인별 영향이 한눈에 보이지 않아 어디에 정비 자원을 먼저 투입할지 정하기 어렵다. 본 프로젝트는 미 국방부 외주정비 품질보증 사례의 문제의식(규격 정합성·검사 병목·하자 연결)을 교육용 가상 자료로 재구성해, 정비 대기 요인 분류, 저가동 요인 분석, 재배치 시뮬레이션, 추세 대시보드, 사람 검토 워크플로우까지 프로토타입으로 구현하는 과제이다. 가상 장비는 유형 A·B·C와 가상 부대로만 구분하며 실제 장비·부대 자료는 쓰지 않는다. 최종 목표는 가동률 저하 요인과 외주 품질검사 병목을 근거와 함께 보여 주되, 정비 우선순위와 자원 투입 결정은 정비관·품질검사관·군수참모가 하도록 지원하는 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

> **💡 [오프라인 폐쇄망 및 로컬 주피터 환경 안내]**
> - 외부 인터넷망(Colab) 외에 군 인트라넷 또는 로컬 환경에서 실행할 경우, 아래 명령어로 필요한 표준 라이브러리를 사전 설치하십시오:
>   `pip install numpy pandas matplotlib scipy scikit-learn`
> - 인터넷이 제한된 환경의 경우 사전 번들링된 Wheel(`.whl`) 패키지 디렉토리를 통해 설치할 수 있습니다:
>   `pip install --no-index --find-links=./wheels/ -r requirements.txt`

> 📁 **[안내: 프로젝트 표준 산출물 디렉터리 가이드]**
> - 본 프로젝트는 단순한 일회성 코드 실습을 넘어 실제 부대에 배포 가능한 업무 도구(Tool)를 완성하는 과정입니다.
> - 실습 중 작성하는 결과물은 역할에 따라 아래 표준 디렉터리에 자동으로 저장·누적됩니다:
>   - `docs/`: 프로세스 정의서(`qlt04_as_is_to_be_spec.md`), UAT 피드백 리포트(`qlt04_uat_feedback_report.md`), 운영 매뉴얼(`qlt04_final_manual.md`) 등 행정·기술 문서
>   - `src/`: 데이터 로더, 16종 품질 판정 룰 엔진, 릿지 회귀 XAI 엔진, 시뮬레이터 UI, HITL 결재 FSM, 배치 러너 등 핵심 실행 모듈
>   - `dist/`: 폐쇄망 및 야전 배포를 위한 최종 패키지(`qlt04_package/`)

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 04. 회귀 분석 및 원인 규명 모델 기반 가동률 저하 핵심 병목 요인 정밀 분석 엔진 고도화
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

회귀 분석 기반 가동률 저하 핵심 병목 요인 정밀 분석 엔진을 고도화한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **회귀 분석으로 가동률 저하의 핵심 병목 요인을 정밀 분석하고, 저하 요인을 분류해 Top-3 개선 옵션과 유사 사례 유사도를 제공하는 모델의 정확도를 검증한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **가상 장비 1,500대를 학습 70%와 검증 30%로 나누고 릿지 회귀로 요인별 표준화 계수를 구한 뒤 부트스트랩 20회로 계수 부호의 안정성 확인하고 유효성을 점검하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/ai/qlt04_intelligence_module.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 추천 사유의 합리성 및 실무 수용성

#### 미션 PASS 조건

- 정량 기준: 추천/분류 정확도 85% 이상 달성


- 진행 기준: AI 추론 모듈 구현 및 벤치마크 테스트 완료


- 검증 방법: 과거 승인/처리 완료 데이터와의 추천 일치도 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""가상 장비 1,500대의 가동률을 릿지 회귀로 설명해 핵심 병목 요인의 영향 크기와 부호 안정성을 분석하고, 저가동 장비의 주요 요인(부품 대기·정비병 부족·외주 정비·검사·재작업)을 분류해 Top-3 개선 옵션과 유사 과거 사례 유사도 점수를 제공하며 정확도를 검증 세트로 측정한다."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_error

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 장비 가동률·정비 대기 자료 생성기 (실제 부대·장비·제원·가동률 정보 없음)
# 장비 유형 A(궤도형)·B(화포형)·C(차륜형)은 추상 구분이며 실제 장비와 무관하다.
# 최근 90일 동안 장비 1대의 정비 대기 요인(부품·정비병·외주 정비·외주 검사·재작업)과 가동률을 만든다.
# ---------------------------------------------------------------------------
UNITS = ["가상A부대", "가상B부대", "가상C부대", "가상D부대", "가상E부대", "가상F부대"]
ETYPES = ["유형A", "유형B", "유형C"]
WINDOW = 90                                       # 관측 기간(일)
CAUSES = ["부품 대기", "정비병 부족", "외주 정비", "검사·재작업"]
CAUSE_COL = {"부품 대기": "parts_wait_days", "정비병 부족": "labor_wait_days", "외주 정비": "outsource_days", "검사·재작업": "inspect_wait_days"}
NUM_FIELDS = ["age_years", "op_hours", "maint_orders", "parts_wait_days", "labor_wait_days", "outsource_days", "inspect_wait_days", "rework_count",
              "spec_mismatch", "defect_reports", "parts_stock_pct", "mechanic_fill_pct", "avail_pct"]
FEAT = ["age_years", "maint_orders", "parts_stock_pct", "mechanic_fill_pct", "outsource_days", "inspect_wait_days", "rework_count", "spec_mismatch", "defect_reports", "etype_B", "etype_C"]
FEAT_KO = {"age_years": "장비 연령", "maint_orders": "정비 요청 건수", "parts_stock_pct": "부품 재고 가용률", "mechanic_fill_pct": "정비병 충원율", "outsource_days": "외주 정비 일수",
           "inspect_wait_days": "외주 검사 대기 일수", "rework_count": "재작업 건수", "spec_mismatch": "규격·제원 불일치", "defect_reports": "하자·불만 접수",
           "etype_B": "유형B 여부", "etype_C": "유형C 여부"}


def unit_profile(seed=3):
    """부대별 고정 특성: 부품 재고 수준, 정비병 충원율, 외주 정비소 거리(일수 가중), 외주 검사 혼잡도."""
    r = np.random.default_rng(seed)
    return pd.DataFrame({"unit": UNITS, "stock": r.uniform(45, 95, len(UNITS)), "fill": r.uniform(55, 98, len(UNITS)),
                         "dist": r.uniform(0.8, 1.6, len(UNITS)), "queue": r.uniform(0.7, 2.2, len(UNITS))})


def make_fleet(n=300, seed=7, shock=0.0):
    """장비 n대의 90일 관측 자료(필드 16개)와 정답 요인을 만든다. shock은 외주 검사 병목 악화 정도(AAR용 환경 변화)."""
    rng = np.random.default_rng(seed)
    pf = unit_profile()
    ui = rng.integers(0, len(UNITS), n)
    et = rng.choice(ETYPES, n, p=[.35, .3, .35])
    age = np.clip(rng.gamma(4, 2.2, n), 0.5, 30).round(1)
    eff = np.array([{"유형A": 1.15, "유형B": 1.0, "유형C": 0.9}[t] for t in et])
    orders = rng.poisson(np.clip(1.2 + 0.12 * age, 0.5, 6) * eff)
    stock = np.clip(pf["stock"].to_numpy()[ui] + rng.normal(0, 8, n), 20, 100).round(0)
    fill = np.clip(pf["fill"].to_numpy()[ui] + rng.normal(0, 6, n), 30, 100).round(0)
    parts_w = (orders * rng.gamma(2, 0.55 * (1 + ((100 - stock) / 28) ** 2) / 2)).round(0)
    labor_w = (orders * rng.gamma(2, 0.7 * (1 + ((100 - fill) / 24) ** 2) / 2)).round(0)
    n_out = rng.binomial(orders, 0.3)
    out_d = (n_out * rng.gamma(3, 4.5 * pf["dist"].to_numpy()[ui] / 3)).round(0)
    insp_w = (n_out * rng.gamma(2, (2.6 + shock) * pf["queue"].to_numpy()[ui] / 2)).round(0)
    spec = rng.poisson(0.2 + 0.04 * age)
    rework = rng.binomial(n_out, np.clip(0.08 + 0.1 * spec, 0, .6))
    defect = rng.poisson(0.3 + 0.7 * rework)
    base_rep = orders * 1.8
    down = base_rep + parts_w + labor_w + out_d + insp_w + rework * 4 + rng.normal(0, 1.2, n)
    comp = np.column_stack([parts_w, labor_w, out_d, insp_w + rework * 4])
    df = pd.DataFrame({"equip_id": [f"E-{i + 1:04d}" for i in range(n)], "unit": np.array(UNITS)[ui], "etype": et, "age_years": age,
                       "op_hours": np.clip(rng.normal(320 - 4 * age, 70, n), 40, 600).round(0), "maint_orders": orders,
                       "parts_wait_days": parts_w, "labor_wait_days": labor_w, "outsource_days": out_d, "inspect_wait_days": insp_w,
                       "rework_count": rework, "spec_mismatch": spec, "defect_reports": defect, "parts_stock_pct": stock, "mechanic_fill_pct": fill,
                       "avail_pct": (100 * (1 - np.clip(down, 0, WINDOW) / WINDOW)).round(1)})
    df["main_cause"] = np.where(comp.max(axis=1) < 3, "없음", np.array(CAUSES)[comp.argmax(axis=1)])
    return df


def feat_frame(df):
    """회귀·분석용 설명 변수 표(유형은 더미 변수)."""
    x = df.copy()
    x["etype_B"], x["etype_C"] = (x["etype"] == "유형B").astype(int), (x["etype"] == "유형C").astype(int)
    return x[FEAT]


def make_trend(months=12, seed=5, shock_unit=None):
    """[교육용가상자료] 부대 x 유형 x 월 평균 가동률 추세. shock_unit 부대는 하반기에 외주 검사 병목으로 가동률이 떨어진다."""
    rng = np.random.default_rng(seed)
    pf = unit_profile()
    rows = []
    for _, u in pf.iterrows():
        base = 62 + 0.22 * u["stock"] + 0.12 * u["fill"] - 3.0 * u["queue"]
        for t in ETYPES:
            off = {"유형A": -3.0, "유형B": 0.0, "유형C": 2.0}[t]
            for m in range(months):
                v = base + off + 2.0 * np.sin(m / 12 * 2 * np.pi) + rng.normal(0, 1.0)
                if shock_unit == u["unit"] and m >= months // 2:
                    v -= 1.5 * (m - months // 2 + 1)
                rows.append((u["unit"], t, m + 1, round(float(np.clip(v, 30, 99)), 1)))
    return pd.DataFrame(rows, columns=["unit", "etype", "month", "avail_pct"])

# ---- 가동률 회귀 모델과 저가동 요인 분류·추천 모델(외부 AI 서비스 없이 scikit-learn 사용) ----
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

CAUSE_COLS = ["age_years", "maint_orders", "parts_stock_pct", "mechanic_fill_pct", "outsource_days", "inspect_wait_days", "rework_count", "spec_mismatch",
              "defect_reports", "parts_wait_days", "labor_wait_days"]
ACTION = {"부품 대기": "부품 사전 청구·재고 보강", "정비병 부족": "정비병 재배치·교육 일정 조정", "외주 정비": "외주 정비 일정 조율·내부 정비 확대", "검사·재작업": "검사 일정 조율·규격 사전 대조"}


def cause_frame(df):
    x = df[CAUSE_COLS].copy()
    x["out_insp"] = x["outsource_days"] + x["inspect_wait_days"] + 4 * x["rework_count"]
    x["wait_pm"] = (x["parts_wait_days"] + x["labor_wait_days"]) / (x["maint_orders"] + 1)
    return x


def fit_avail(df, alpha=1.0):
    """
    [TODO : 학생 실습 4-1] 설명 변수들로 장비 가동률(%)을 예측하는 릿지(Ridge) 회귀 파이프라인을 구축하고 학습하세요.
    - 힌트: make_pipeline(StandardScaler(), Ridge(alpha=alpha))을 사용하여 feat_frame(df)와 df["avail_pct"]를 적합(fit)합니다.
    """
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.


def fit_cause(df, cut=85, C=5):
    """
    [TODO : 학생 실습 4-2] 저가동 장비(가동률 cut% 미만)의 병목 원인을 분류하는 로지스틱 회귀 모델을 학습하세요.
    - 힌트 1: low = df[(df["avail_pct"] < cut) & (df["main_cause"] != "없음")]로 저가동 대상을 필터링합니다.
    - 힌트 2: make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000))을 cause_frame(low)와 low["main_cause"]에 적합합니다.
    """
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.


SIM_FEAT = ["age_years", "maint_orders", "spec_mismatch", "stock_gap_x_orders", "fill_gap_x_orders", "parts_stock_pct", "mechanic_fill_pct", "etype_B", "etype_C"]


def sim_frame(df):
    """자원 재배치 시뮬레이션용 설명 변수: 부족분이 클수록 대기가 더 빠르게 늘고 정비 요청 건수와 곱해져 커진다는 구조를 제곱·상호작용 항으로 둔다."""
    x = df.copy()
    x["etype_B"], x["etype_C"] = (x["etype"] == "유형B").astype(int), (x["etype"] == "유형C").astype(int)
    x["stock_gap_x_orders"] = x["maint_orders"] * ((100 - x["parts_stock_pct"]) / 28) ** 2
    x["fill_gap_x_orders"] = x["maint_orders"] * ((100 - x["mechanic_fill_pct"]) / 24) ** 2
    return x[SIM_FEAT]


def fit_sim(df, alpha=0.5):
    return make_pipeline(StandardScaler(), Ridge(alpha=alpha)).fit(sim_frame(df), df["avail_pct"])

fleet = make_fleet(1500, seed=21)
idx = np.random.default_rng(5).permutation(len(fleet))
tr, te = fleet.iloc[idx[:1050]], fleet.iloc[idx[1050:]]                     # 학습 70% / 검증 30%(과거 처리 완료 자료 가정)

# 1) 회귀 분석: 가동률 = f(연령, 정비 요청, 재고, 충원율, 외주·검사 대기, 재작업, 규격 불일치, 하자, 유형)
reg = fit_avail(tr)
pred = reg.predict(feat_frame(te))
coef = pd.Series(reg[-1].coef_, index=FEAT)                                # 표준화 계수: 1표준편차 증가 시 가동률 변화(%p)
print(f"[회귀 모델] 학습 {len(tr)}대 / 검증 {len(te)}대: 검증 R2 {r2_score(te['avail_pct'], pred):.3f}, MAE {mean_absolute_error(te['avail_pct'], pred):.2f}%p")
boot = []
rng = np.random.default_rng(9)
for _ in range(20):                                                       # 부트스트랩 20회로 계수 부호 안정성 확인
    b = tr.iloc[rng.integers(0, len(tr), len(tr))]
    boot.append(fit_avail(b)[-1].coef_)
boot = np.array(boot)
top = coef.abs().sort_values(ascending=False).head(5).index
print("[핵심 병목 요인: 표준화 계수(1 표준편차 증가 시 가동률 변화 %p) / 부호 유지 비율]")
for f in top:
    k = FEAT.index(f)
    print(f"  {FEAT_KO[f]:12s} {coef[f]:+6.2f}%p  부호 유지 {np.mean(np.sign(boot[:, k]) == np.sign(coef[f])):.0%}")

# 2) 요인 분류 + Top-3 개선 옵션
clf = fit_cause(tr)
low = te[(te["avail_pct"] < 85) & (te["main_cause"] != "없음")]
X = cause_frame(low)
proba = clf.predict_proba(X)
cls = clf.classes_
top1 = cls[proba.argmax(1)]
acc1 = float(np.mean(top1 == low["main_cause"].to_numpy()))
order = np.argsort(-proba, axis=1)[:, :3]
acc3 = float(np.mean([low["main_cause"].iloc[i] in cls[order[i]] for i in range(len(low))]))
major = low["main_cause"].value_counts(normalize=True).iloc[0]
print(f"\n[요인 분류] 검증 저가동 장비 {len(low)}대: Top-1 정확도 {acc1:.1%}, Top-3 포함률 {acc3:.1%} (무작위 최빈값 기준 {major:.1%}) -> 목표 85% 이상: {'달성' if acc1 >= 0.85 else '미달'}")
rc = pd.crosstab(low["main_cause"], pd.Series(top1, index=low.index, name="예측"))
print("요인별 재현율:", ", ".join(f"{c} {rc.loc[c, c] / rc.loc[c].sum():.0%}" for c in rc.index if c in rc.columns))

# 3) 유사 과거 사례 유사도(코사인): 학습 세트 중 같은 요인 장비와 얼마나 비슷한가
sc, trl = clf[0], tr[(tr["avail_pct"] < 85) & (tr["main_cause"] != "없음")]
Z = sc.transform(cause_frame(trl)); Zn = Z / np.linalg.norm(Z, axis=1, keepdims=True)
zq = sc.transform(X); zqn = zq / np.linalg.norm(zq, axis=1, keepdims=True)
sim = {c: (zqn @ Zn[(trl["main_cause"] == c).to_numpy()].T).max(axis=1) for c in cls}
print("\n[Top-3 개선 옵션 예시: 확률(%) / 유사 사례 유사도]")
for j in range(3):
    r = low.iloc[j]
    opts = " | ".join(f"{cls[o]} {proba[j, o]:.0%}/{sim[cls[o]][j]:.2f} -> {ACTION[cls[o]]}" for o in order[j])
    print(f"  {r['equip_id']} 가동률 {r['avail_pct']}% (정답 {r['main_cause']}): {opts}")
miss = low[top1 != low["main_cause"].to_numpy()]
print(f"\n[오분류 {len(miss)}건] 확률이 가장 높은 요인과 2위 확률 차이 평균 {np.mean(np.sort(proba, axis=1)[:, -1] - np.sort(proba, axis=1)[:, -2]):.2f}")
conf = np.sort(proba, axis=1)[:, -1]
print(f"확률 0.8 이상 추천의 정확도 {np.mean((top1 == low['main_cause'].to_numpy())[conf >= 0.8]):.0%} ({int((conf >= 0.8).sum())}건), 0.8 미만은 사람이 Top-3를 비교해 선택")
print("※ 추천은 판단 보조이며 정비 우선순위와 자원 투입은 정비관·군수참모가 결정한다. 가상 자료이므로 실제 장비에서는 정확도가 다를 수 있다.")

plt.figure(figsize=(7, 4))
plt.barh(list(coef.abs().sort_values().index), coef[coef.abs().sort_values().index], color="tab:blue")
plt.title("Standardized ridge coefficients on availability (%p per 1 SD)"); plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 04 예상 실행 결과 (Output)</b></summary>

```text
[회귀 모델] 학습 1050대 / 검증 450대: 검증 R2 0.899, MAE 3.97%p
[핵심 병목 요인: 표준화 계수(1 표준편차 증가 시 가동률 변화 %p) / 부호 유지 비율]
  정비 요청 건수      -8.63%p  부호 유지 100%
  외주 정비 일수      -6.62%p  부호 유지 100%
  외주 검사 대기 일수   -5.06%p  부호 유지 100%
  재작업 건수        -1.88%p  부호 유지 100%
  부품 재고 가용률     +1.73%p  부호 유지 100%

[요인 분류] 검증 저가동 장비 234대: Top-1 정확도 99.1%, Top-3 포함률 100.0% (무작위 최빈값 기준 35.5%) -> 목표 85% 이상: 달성
요인별 재현율: 검사·재작업 97%, 부품 대기 100%, 외주 정비 99%, 정비병 부족 100%

[Top-3 개선 옵션 예시: 확률(%) / 유사 사례 유사도]
  E-1269 가동률 79.4% (정답 검사·재작업): 검사·재작업 98%/0.89 -> 검사 일정 조율·규격 사전 대조 | 외주 정비 2%/0.83 -> 외주 정비 일정 조율·내부 정비 확대 | 정비병 부족 0%/0.68 -> 정비병 재배치·교육 일정 조정
  E-0462 가동률 74.4% (정답 외주 정비): 외주 정비 63%/0.74 -> 외주 정비 일정 조율·내부 정비 확대 | 부품 대기 36%/0.81 -> 부품 사전 청구·재고 보강 | 정비병 부족 0%/0.69 -> 정비병 재배치·교육 일정 조정
  E-0924 가동률 3.6% (정답 외주 정비): 외주 정비 100%/0.85 -> 외주 정비 일정 조율·내부 정비 확대 | 부품 대기 0%/0.70 -> 부품 사전 청구·재고 보강 | 정비병 부족 0%/0.75 -> 정비병 재배치·교육 일정 조정

[오분류 2건] 확률이 가장 높은 요인과 2위 확률 차이 평균 0.86
확률 0.8 이상 추천의 정확도 100% (192건), 0.8 미만은 사람이 Top-3를 비교해 선택
※ 추천은 판단 보조이며 정비 우선순위와 자원 투입은 정비관·군수참모가 결정한다. 가상 자료이므로 실제 장비에서는 정확도가 다를 수 있다.
```
</details>
